In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [4]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


#### Train test devide


In [5]:
x_train, x_test, y_train, y_test = train_test_split(df.drop(columns=['diagnosis']), df['diagnosis'], test_size=0.2, random_state=42)

In [6]:
# scaling
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


In [7]:
# label encoder
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)  


Numpy array to pytorch

In [8]:
x_train_tensor = torch.from_numpy(x_train)
x_test_tensor = torch.from_numpy(x_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [9]:
y_train_tensor.shape

torch.Size([455])

## define model

In [17]:
class SimpleNN():
    def __init__(self, X):
        self.weights = torch.randn(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def compute_loss(self, y_pred, y):
        loss = - (y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
        return loss
    
        


In [18]:
learning_rate = 0.01
epochs = 25


# Traning pipeline

In [19]:
model = SimpleNN(x_train_tensor)

In [20]:
# define epochs and learning rate
for epoch in range(epochs):
# Forward pass
    y_pred = model.forward(x_train_tensor)
    # print(y_pred)

# loss calculation
    loss = model.compute_loss(y_pred, y_train_tensor)
# Backward pass
    loss.backward()
# perameter update
    with torch.no_grad():   
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

# zero the gradients after updating
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')




Epoch 1/25, Loss: 2.6485757096167197
Epoch 2/25, Loss: 2.6352686635406353
Epoch 3/25, Loss: 2.6220206026196125
Epoch 4/25, Loss: 2.6088319195554903
Epoch 5/25, Loss: 2.595703006841991
Epoch 6/25, Loss: 2.5826342797977437
Epoch 7/25, Loss: 2.5696261300216374
Epoch 8/25, Loss: 2.5566789479083414
Epoch 9/25, Loss: 2.543793145965359
Epoch 10/25, Loss: 2.530969087596301
Epoch 11/25, Loss: 2.5182071811850015
Epoch 12/25, Loss: 2.505507808673293
Epoch 13/25, Loss: 2.4928713486923058
Epoch 14/25, Loss: 2.4802981760800527
Epoch 15/25, Loss: 2.4677886856301416
Epoch 16/25, Loss: 2.4553432432293927
Epoch 17/25, Loss: 2.442962184934183
Epoch 18/25, Loss: 2.4306458894941017
Epoch 19/25, Loss: 2.4183946802837375
Epoch 20/25, Loss: 2.4062089224149443
Epoch 21/25, Loss: 2.3940889487698134
Epoch 22/25, Loss: 2.3820350837701487
Epoch 23/25, Loss: 2.370047642730857
Epoch 24/25, Loss: 2.3581269065829247
Epoch 25/25, Loss: 2.346273195073401


In [21]:
# model evaluation
with torch.no_grad():
    y_pred_test = model.forward(x_test_tensor)
    y_pred_test = (y_pred_test > 0.9).float()
    accuracy = (y_pred_test == y_test_tensor).float().mean()
    print(f'Test Accuracy: {accuracy.item()}')

Test Accuracy: 0.5150815844535828


Make model wuth NN


In [24]:
from torch import nn
class SimpleNN(torch.nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out

In [25]:
learning_rate = 0.01
epochs = 25

In [26]:
loss_fn = nn.BCELoss()

In [27]:
model = SimpleNN(x_train_tensor.shape[1])

In [37]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [38]:
# define epochs and learning rate
for epoch in range(epochs):
# Forward pass
    y_pred = model(x_train_tensor.float())

    # print(y_pred)

# loss calculation
    loss = loss_fn(y_pred, y_train_tensor.float().view(-1, 1))
# clear the gradients before running the backward pass
    optimizer.zero_grad()
# Backward pass
    loss.backward()
# perameter update
    optimizer.step()

    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')




Epoch 1/25, Loss: 0.3480932414531708
Epoch 2/25, Loss: 0.3456402122974396
Epoch 3/25, Loss: 0.3432405889034271
Epoch 4/25, Loss: 0.340892493724823
Epoch 5/25, Loss: 0.3385940492153168
Epoch 6/25, Loss: 0.3363436162471771
Epoch 7/25, Loss: 0.33413955569267273
Epoch 8/25, Loss: 0.33198022842407227
Epoch 9/25, Loss: 0.3298642635345459
Epoch 10/25, Loss: 0.32779011130332947
Epoch 11/25, Loss: 0.3257564604282379
Epoch 12/25, Loss: 0.32376205921173096
Epoch 13/25, Loss: 0.3218056261539459
Epoch 14/25, Loss: 0.3198859393596649
Epoch 15/25, Loss: 0.3180018961429596
Epoch 16/25, Loss: 0.3161523938179016
Epoch 17/25, Loss: 0.3143364191055298
Epoch 18/25, Loss: 0.31255292892456055
Epoch 19/25, Loss: 0.3108009696006775
Epoch 20/25, Loss: 0.3090796172618866
Epoch 21/25, Loss: 0.3073880672454834
Epoch 22/25, Loss: 0.3057253658771515
Epoch 23/25, Loss: 0.3040907680988312
Epoch 24/25, Loss: 0.3024834394454956
Epoch 25/25, Loss: 0.3009026348590851


In [35]:
# model evaluation
with torch.no_grad():
    y_pred_test = model.forward(x_test_tensor.float())
    y_pred_test = (y_pred_test > 0.9).float()
    accuracy = (y_pred_test == y_test_tensor).float().mean()
    print(f'Test Accuracy: {accuracy.item()}')

Test Accuracy: 0.588334858417511
